# SQG data assimilation demo: four observation modes + parameter sweeps + relaxed residual normalization + localized weighted $J_F$

Workflow:

1. **Burn-in** — true SQG solver spin-up (physical PV).
2. **Load model** — conditional diffusion checkpoint + norm stats.
3. **Observation + EDM settings** — mixed direct/atan observations and EDM sampler.
4. **DA cycles** — history-conditioned analysis with relaxed residual normalization and a localized weighted ensemble approximation of the denoiser Jacobian action.
5. **Sweep** — reproducible parameter sweep for any of the four observation modes, automatic best-case selection, then a full rerun of the selected case.

Truth solver stays physical; normalize at observation time. Denormalize only for plots/saves.

The four supported observation modes are `random_percent`, `kstep_nonoverlap`, `block`, and `diagonal`.

The denoiser is written in EDM form

$$
\hat x_0
=
c_{\rm skip}x
+
c_{\rm out}F(c_{\rm in}x,H),
$$

with the history $H$ held as the model condition. The likelihood direction is transported with

$$
J_D^Tg
=
c_{\rm skip}g
+
c_{\rm out}c_{\rm in}J_F^Tg.
$$

Here $J_F^Tg$ is estimated from the ensemble using periodic 2D local regressions. For each observed output pixel, the local input patch contains **both SQG layers**.


In [ ]:
import os
import pickle
import time
from itertools import product

import dnnlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import tqdm
from tqdm.autonotebook import tqdm as notebook_tqdm
import math
from project_utils.normalization import (
    denormalize_tensor,
    load_norm_stats,
    normalize_tensor,
    resolve_norm_stats_path,
)
from project_utils.paths import PROJECT_ROOT, dataset_trajectory_dir, CKPT_DIR
from project_utils.yaml_simple import load_simple_yaml
from turbulence.sqg import SQGParams, SQGSolver

### Utilities

Shared helpers for burn-in, ensemble histories, and the DA loop.

**Shape convention:** `C=2` PV layers, `H=W=NX`, `Ne=NE`. Single frame: `(C, H, W)`.

**Normalized DA:** after burn-in, ensemble histories store **normalized** PV in [-1, 1]. DM conditions are built directly from those frames (no extra normalize step).


In [ ]:
class StackedRandomGenerator:
    """One RNG per ensemble member; ``randn([Ne, ...])`` stacks independent draws."""

    def __init__(self, device, seeds):
        self.generators = [
            torch.Generator(device).manual_seed(int(s) % (1 << 32)) for s in seeds
        ]

    def randn(self, size, **kwargs):
        # size[0]=Ne; returns (Ne, *size[1:])
        if len(size) < 1 or len(self.generators) != size[0]:
            raise ValueError(
                f"size[0]={size[0] if size else None} must match n_generators={len(self.generators)}"
            )
        return torch.stack(
            [torch.randn(size[1:], generator=g, **kwargs) for g in self.generators]
        )

    def randn_like(self, input):
        return self.randn(
            input.shape, dtype=input.dtype, layout=input.layout, device=input.device
        )


def make_initial_pv(N, device, seed):
    torch.manual_seed(seed)
    pv = torch.randn(2, N, N, dtype=torch.float32, device=device) * 100.0  # (C, H, W)
    nexp = 20
    x = torch.arange(0.0, 2.0 * np.pi, 2.0 * np.pi / N, device=device, dtype=torch.float32)
    y = torch.arange(0.0, 2.0 * np.pi, 2.0 * np.pi / N, device=device, dtype=torch.float32)
    x, y = torch.meshgrid(x, y, indexing="xy")  # (H, W) each
    pv[1] = pv[1] + 2000.0 * (torch.sin(x / 2.0) ** (2 * nexp) * torch.sin(y) ** nexp)
    return pv - pv.mean(dim=(1, 2), keepdim=True)  # (C, H, W)


def build_condition_from_history(frames_norm, history):
    # frames_norm: list of (C, H, W) or (T, C, H, W), already normalized -> (1, history*C, H, W)
    if isinstance(frames_norm, torch.Tensor):
        block = frames_norm[-history:]  # (history, C, H, W)
    else:
        block = torch.stack(frames_norm[-history:], dim=0)
    C, H, W = block.shape[1], block.shape[2], block.shape[3]
    return block.reshape(history * C, H, W).unsqueeze(0)


def build_condition_from_history_batch(histories, history):
    """Stack conditions for each ensemble member: ``(Ne, cond_ch, H, W)`` (normalized)."""
    return torch.cat(
        [build_condition_from_history(h, history) for h in histories],
        dim=0,
    )


def perturb_norm(x_norm, std, seed):
    # x_norm: (C, H, W) in [-1, 1]
    g = torch.Generator().manual_seed(int(seed) % (1 << 32))
    return x_norm + std * torch.randn(
        x_norm.shape, generator=g, dtype=x_norm.dtype, device=x_norm.device
    )


def ensemble_batch_rmse(ens, truth_norm):
    """RMSE/spread of ensemble mean vs truth. ens: (Ne, C, H, W)."""
    mean = ens.mean(dim=0)
    spread = ens.std(dim=0, correction=0).mean().item() if ens.shape[0] > 1 else 0.0
    rmse = torch.sqrt(torch.mean((mean - truth_norm) ** 2)).item()
    return rmse, spread


def ensemble_rmse_spatial_split(ens, truth_norm, obs_mask):
    """RMSE of ensemble mean on full grid, observed sites, and unobserved sites."""
    mean = ens.mean(dim=0)
    err2 = (mean - truth_norm).pow(2)
    m = obs_mask.to(device=err2.device, dtype=torch.bool)
    n_obs_sites = int(m.sum().item())
    n_total = m.numel()
    rmse_all = torch.sqrt(err2.mean()).item()
    rmse_obs = (
        torch.sqrt(err2[m].mean()).item() if n_obs_sites > 0 else float("nan")
    )
    rmse_unobs = (
        torch.sqrt(err2[~m].mean()).item()
        if n_obs_sites < n_total
        else float("nan")
    )
    return rmse_all, rmse_obs, rmse_unobs


def ens_channel_phys(path_ens, member, t_slice, channel, stats):
    """Denorm one member/time slice for plots. path_ens: (Ne,T,C,H,W) -> (T,H,W)."""
    phys = denormalize_tensor(path_ens[member, t_slice], stats)
    return phys[:, channel]


In [ ]:
# --- User configuration ---
SQG_CONFIG = os.path.join(PROJECT_ROOT, "configs/turbulence/sqg_64d_1h.yaml")
DATA_ROOT = "SQG_64d_1h"
TRAJECTORY_SUBDIR = "full_path"

NETWORK_PKL = os.path.join(
    CKPT_DIR,
    "sqg_dim_64_dt_1.pkl",
)

DEVICE = "cuda:0"
SEED = 0
SEED_DM = 1
SEED_Forecast = 2

NX = 64  # grid size H = W = NX
T_BURN_IN = 400
N_DA_CYCLES = 250
OUTPUT_INTERVAL_HOURS = 1.0

HISTORY = 1  # past frames fed as DM condition per member

# --- Ensemble DA (all in normalized [-1, 1] space after init) ---
NE = 36  # number of ensemble members; set to 1 for single-member run
ENSEMBLE_PERTURB_STD = 0.01  # perturbation std on last burn-in frame (normalized units)

device = torch.device(DEVICE)
trajectory_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
norm_stats_path = resolve_norm_stats_path(trajectory_dir, None)

## Phase 1: True SQG burn-in

In [ ]:
cfg = load_simple_yaml(SQG_CONFIG)
sqg_cfg = dict(cfg.get("sqg", {}))
sqg_cfg["device"] = str(device)

params = SQGParams(**sqg_cfg)
params.validate(N=NX)
timesteps_per_output = max(1, int(OUTPUT_INTERVAL_HOURS * 3600.0 / params.dt))

pv0 = make_initial_pv(NX, device, SEED)  # (C, H, W) = (2, NX, NX)
solver = SQGSolver(pv0, **params.to_core_kwargs())

burn_frames = [solver.fields()["pv"].detach().cpu()]  # list of (C, H, W)
for _ in tqdm.tqdm(range(T_BURN_IN), desc="SQG burn-in"):
    solver.step(timesteps_per_output)
    burn_frames.append(solver.fields()["pv"].detach().cpu())

state_after_burn = {k: v.clone() for k, v in solver.get_state().items()}  # solver internal state
path_burn = torch.stack(burn_frames, dim=0)  # (T_burn+1, C, H, W)
print("Burn-in frames:", tuple(path_burn.shape))

## Phase 2: Load trained conditional diffusion model + norm stats

In [ ]:
if not os.path.isfile(NETWORK_PKL):
    raise FileNotFoundError(f"Set NETWORK_PKL. Not found: {NETWORK_PKL}")

norm_stats = load_norm_stats(norm_stats_path)  # dict: vmin/vmax (global or per-channel)
with dnnlib.util.open_url(NETWORK_PKL, verbose=True) as f:
    net = pickle.load(f)["ema"].to(device).eval()

n_target = 2  # PV channels predicted by DM
n_cond = HISTORY * n_target  # condition channels = history * C
assert net.img_input_channels == n_target
assert net.img_cond_channels == n_cond
assert net.img_res_h == NX and net.img_res_w == NX

print(f"Loaded {NETWORK_PKL}")

## Phase 3: Initialize for DA

Ensemble histories and metric containers are reset inside the **Phase 4** cell each run, so you can re-run that single cell to repeat the experiment.


### Observation operator (normalized space)

Two parts: **elementwise map** \(g(x)\) on masked sites, then **mask** selects where \(y\) is defined.

$$y = g(x[\texttt{obs\_mask}]) + \epsilon,\quad \epsilon \sim \mathcal{N}(0, \sigma_{\mathrm{obs}}^2)$$


In [ ]:
def make_fixed_obs_indices_equal_channels(
    Nx, obs_percent_fix, obs_channels, seed_fix=0):
    rng = np.random.default_rng(seed_fix)

    n_grid = Nx * Nx
    n_fix_per_channel = int(round(obs_percent_fix / 100 * n_grid))

    fixed_by_channel = {}

    for c in obs_channels:
        base = int(c) * n_grid
        channel_indices = np.arange(base, base + n_grid, dtype=np.int64)

        fixed_idx = rng.choice(
            channel_indices,
            size=n_fix_per_channel,
            replace=False,
        )

        fixed_by_channel[int(c)] = fixed_idx

    return fixed_by_channel

In [ ]:
def n_obs(obs_mask):
    return int(obs_mask.sum().item())


def check_obs_mask(obs_mask, n_channels, nx):
    """Validate obs_mask shape and that at least one site is observed."""
    if tuple(obs_mask.shape) != (n_channels, nx, nx):
        raise ValueError(f"obs_mask shape {tuple(obs_mask.shape)} != ({n_channels}, {nx}, {nx})")
    if obs_mask.dtype != torch.bool:
        raise ValueError("obs_mask must be bool")
    n = n_obs(obs_mask)
    if n == 0:
        raise ValueError("obs_mask has no True sites")
    print(f"obs_mask OK: n_obs={n}, shape=({n_channels}, {nx}, {nx})")

def combine_fixed_and_random_obs_indices_equal_channels(
    Nx, obs_percent_random, obs_channels, fixed_by_channel, seed_random=0, device="cpu"):
    rng = np.random.default_rng(seed_random)

    n_grid = Nx * Nx
    n_random_per_channel = int(round(obs_percent_random / 100 * n_grid))

    flat_indices_all = []

    for c in obs_channels:
        c = int(c)
        base = c * n_grid
        channel_indices = np.arange(base, base + n_grid, dtype=np.int64)

        fixed_idx = fixed_by_channel[c]

        remaining_idx = np.setdiff1d(
            channel_indices, fixed_idx, assume_unique=False)

        if n_random_per_channel > remaining_idx.size:
            raise ValueError(
                "obs_percent_random is too large after removing fixed locations."
            )

        random_idx = rng.choice(
            remaining_idx, size=n_random_per_channel, replace=False)

        flat_indices_all.append(np.concatenate([fixed_idx, random_idx]))

    flat_idx = np.concatenate(flat_indices_all)
    rng.shuffle(flat_idx)

    return torch.tensor(flat_idx, dtype=torch.long, device=device)

def obs_mask_vector(flat_idx, total_size, device=None, dtype=torch.float32):
    """Dense mask m in {0,1}^N with y_i = x_i where m_i = 1."""
    dev = device if device is not None else flat_idx.device
    m = torch.zeros(total_size, device=dev, dtype=dtype)
    m[flat_idx.to(dev)] = 1.0
    return m

def fixed_obs_indices_equal_channels(fixed_by_channel, obs_channels, device="cpu"):
    """Flatten fixed observation indices from fixed_by_channel."""
    fixed_all = []
    for c in obs_channels:
        fixed_all.append(np.asarray(fixed_by_channel[int(c)], dtype=np.int64))
    fixed_idx = np.concatenate(fixed_all)
    return torch.tensor(fixed_idx, dtype=torch.long, device=device)


def random_obs_indices_excluding_fixed_equal_channels(
    Nx, obs_percent_random, obs_channels, fixed_by_channel, seed_random=0, device="cpu"):
    """Random-only observation indices, excluding the fixed linear sites."""
    rng = np.random.default_rng(seed_random)

    n_grid = Nx * Nx
    n_random_per_channel = int(round(obs_percent_random / 100 * n_grid))

    random_indices_all = []

    for c in obs_channels:
        c = int(c)
        base = c * n_grid
        channel_indices = np.arange(base, base + n_grid, dtype=np.int64)

        fixed_idx = np.asarray(fixed_by_channel[c], dtype=np.int64)

        remaining_idx = np.setdiff1d(
            channel_indices, fixed_idx, assume_unique=False)

        if n_random_per_channel > remaining_idx.size:
            raise ValueError(
                "obs_percent_random is too large after removing fixed locations."
            )

        random_idx = rng.choice(
            remaining_idx, size=n_random_per_channel, replace=False)

        random_indices_all.append(random_idx)

    random_idx = np.concatenate(random_indices_all)
    rng.shuffle(random_idx)

    return torch.tensor(random_idx, dtype=torch.long, device=device)


def _candidate_flat_indices_for_channel(Nx, channel, fixed_by_channel=None, exclude_fixed=True):
    """All flat indices in one channel, optionally excluding the fixed linear sites."""
    n_grid = Nx * Nx
    channel = int(channel)
    base = channel * n_grid
    channel_indices = np.arange(base, base + n_grid, dtype=np.int64)

    if exclude_fixed and fixed_by_channel is not None:
        fixed_idx = np.asarray(fixed_by_channel[channel], dtype=np.int64)
        channel_indices = np.setdiff1d(channel_indices, fixed_idx, assume_unique=False)

    return channel_indices


def _spatial_to_flat_indices(Nx, spatial_ij, obs_channels, device="cpu"):
    """
    Convert spatial indices (i,j) to flattened indices for all selected channels.
    spatial_ij: array-like of shape (n_sites, 2), with 0 <= i,j < Nx.
    """
    spatial_ij = np.asarray(spatial_ij, dtype=np.int64)
    if spatial_ij.size == 0:
        return torch.empty(0, dtype=torch.long, device=device)

    n_grid = Nx * Nx
    flat_all = []

    for c in obs_channels:
        base = int(c) * n_grid
        flat_c = base + spatial_ij[:, 0] * Nx + spatial_ij[:, 1]
        flat_all.append(flat_c)

    flat_idx = np.concatenate(flat_all)
    return torch.tensor(flat_idx, dtype=torch.long, device=device)


def _drop_fixed_from_flat_indices(flat_idx_np, fixed_by_channel, obs_channels):
    """Remove fixed linear indices from a flat-index array."""
    fixed_all = []
    for c in obs_channels:
        fixed_all.append(np.asarray(fixed_by_channel[int(c)], dtype=np.int64))
    fixed_all = np.concatenate(fixed_all)

    flat_idx_np = np.asarray(flat_idx_np, dtype=np.int64)
    return np.setdiff1d(flat_idx_np, fixed_all, assume_unique=False)


def kstep_nonoverlap_obs_indices_equal_channels(
    Nx,
    obs_percent_random,
    obs_max,
    obs_channels,
    fixed_by_channel,
    cycle,
    k_nonoverlap=8,
    base_seed=0,
    device="cpu",
):
    if k_nonoverlap <= 0:
        raise ValueError("k_nonoverlap must be positive.")

    rng = np.random.default_rng(int(base_seed) + int(cycle) // int(k_nonoverlap))

    n_grid = Nx * Nx

    # Actual number used at this filtering step
    n_random_per_channel = int(round(obs_percent_random / 100.0 * n_grid))

    # Reserved slot size, based on the maximum possible percentage
    slot_size = int(math.ceil(obs_max / 100.0 * n_grid))

    if n_random_per_channel > slot_size:
        raise ValueError(
            f"Requested {n_random_per_channel} observations, "
            f"but slot size is only {slot_size}. Check obs_percent_random <= obs_max."
        )

    slot = int(cycle) % int(k_nonoverlap)

    random_indices_all = []

    for c in obs_channels:
        remaining_idx = _candidate_flat_indices_for_channel(
            Nx,
            c,
            fixed_by_channel=fixed_by_channel,
            exclude_fixed=True,
        )

        total_needed = slot_size * int(k_nonoverlap)

        if total_needed > remaining_idx.size:
            raise ValueError(
                "k-step nonoverlap setting requests too many random sites. "
                f"Need {total_needed} available sites per channel, but only "
                f"{remaining_idx.size} remain after excluding fixed sites."
            )

        perm = rng.permutation(remaining_idx)

        start = slot * slot_size
        stop = start + slot_size

        slot_indices = perm[start:stop]

        # Take only the actual requested number for this cycle
        random_indices_all.append(slot_indices[:n_random_per_channel])

    random_idx = np.concatenate(random_indices_all)

    return torch.tensor(random_idx, dtype=torch.long, device=device)


# def block_rect_obs_indices_equal_channels(
#     Nx,
#     obs_channels,
#     fixed_by_channel,
#     n_blocks=10,
#     block_shape=(4, 4),
#     seed_random=0,
#     device="cpu",
#     max_attempts=50000,
# ):
#     """
#     Random nonlinear indices from disjoint rectangular spatial blocks.

#     The same spatial blocks are used for all selected channels. Fixed linear
#     sites are excluded from the returned random nonlinear indices.

#     block_shape can be (M, M) or (M, N).
#     """
#     rng = np.random.default_rng(seed_random)

#     if isinstance(block_shape, int):
#         block_h = int(block_shape)
#         block_w = int(block_shape)
#     else:
#         block_h, block_w = map(int, block_shape)

#     if block_h <= 0 or block_w <= 0:
#         raise ValueError("block_shape entries must be positive.")
#     if block_h > Nx or block_w > Nx:
#         raise ValueError("block_shape must fit inside the Nx by Nx grid.")

#     occupied = np.zeros((Nx, Nx), dtype=bool)
#     blocks = []
#     attempts = 0

#     while len(blocks) < int(n_blocks) and attempts < int(max_attempts):
#         attempts += 1
#         i0 = int(rng.integers(0, Nx - block_h + 1))
#         j0 = int(rng.integers(0, Nx - block_w + 1))

#         if occupied[i0:i0 + block_h, j0:j0 + block_w].any():
#             continue

#         occupied[i0:i0 + block_h, j0:j0 + block_w] = True
#         blocks.append((i0, j0))

#     if len(blocks) < int(n_blocks):
#         raise ValueError(
#             f"Could only place {len(blocks)} disjoint blocks out of requested "
#             f"{n_blocks}. Try smaller n_blocks or block_shape."
#         )

#     spatial_sites = np.argwhere(occupied)
#     flat_idx = _spatial_to_flat_indices(
#         Nx,
#         spatial_sites,
#         obs_channels=obs_channels,
#         device="cpu",
#     ).cpu().numpy()

#     flat_idx = _drop_fixed_from_flat_indices(
#         flat_idx,
#         fixed_by_channel=fixed_by_channel,
#         obs_channels=obs_channels,
#     )

#     return torch.tensor(flat_idx, dtype=torch.long, device=device)

def block_rect_obs_indices_equal_channels(
    Nx,
    obs_channels,
    fixed_by_channel,
    n_blocks=10,
    block_shape=(4, 4),
    seed_random=0,
    device="cpu",
    max_attempts=50000,
):
    rng = np.random.default_rng(seed_random)

    if isinstance(block_shape, int):
        block_h = int(block_shape)
        block_w = int(block_shape)
    else:
        block_h, block_w = map(int, block_shape)

    if block_h <= 0 or block_w <= 0:
        raise ValueError("block_shape entries must be positive.")

    if block_h > Nx or block_w > Nx:
        raise ValueError(
            "block_shape must fit inside the Nx by Nx grid."
        )

    n_grid = Nx * Nx
    all_flat_indices = []

    # Generate an independent block configuration for each channel.
    for c in obs_channels:
        c = int(c)

        occupied = np.zeros((Nx, Nx), dtype=bool)
        blocks = []
        attempts = 0

        while (
            len(blocks) < int(n_blocks)
            and attempts < int(max_attempts)
        ):
            attempts += 1

            i0 = int(rng.integers(0, Nx - block_h + 1))
            j0 = int(rng.integers(0, Nx - block_w + 1))

            block_region = occupied[
                i0:i0 + block_h,
                j0:j0 + block_w,
            ]

            # Prevent overlap between blocks in the same channel.
            if block_region.any():
                continue

            occupied[
                i0:i0 + block_h,
                j0:j0 + block_w,
            ] = True

            blocks.append((i0, j0))

        if len(blocks) < int(n_blocks):
            raise ValueError(
                f"Channel {c}: could only place {len(blocks)} "
                f"disjoint blocks out of the requested {n_blocks}. "
                "Try smaller n_blocks or block_shape."
            )

        # All boundary and interior points in this channel's blocks.
        spatial_sites = np.argwhere(occupied)

        # Convert this channel's spatial sites to global flat indices.
        channel_indices = (
            c * n_grid
            + spatial_sites[:, 0] * Nx
            + spatial_sites[:, 1]
        ).astype(np.int64)

        # Remove fixed linear indices for this particular channel.
        if fixed_by_channel is not None and c in fixed_by_channel:
            fixed_idx = np.asarray(
                fixed_by_channel[c],
                dtype=np.int64,
            )

            channel_indices = np.setdiff1d(
                channel_indices,
                fixed_idx,
                assume_unique=False,
            )

        all_flat_indices.append(channel_indices)

    all_flat_indices = np.concatenate(all_flat_indices)

    return torch.tensor(
        all_flat_indices,
        dtype=torch.long,
        device=device,
    )
def random_k_diagonal_obs_indices_equal_channels(
    Nx,
    n_diagonals,
    obs_channels,
    seed_random,
    device="cpu",
    periodic=True,
    fixed_by_channel=None,
):
    """
    Randomly choose K diagonals at each filtering step.

    For each selected diagonal offset d, we observe grid points
        j = i + d
    or, if periodic=True,
        j = (i + d) mod Nx.

    The percentage per channel is approximately:
        n_diagonals / Nx
    for periodic=True.

    fixed_by_channel is optional. If provided, selected diagonal indices
    that overlap with fixed indices are removed.
    """

    rng = np.random.default_rng(seed_random)

    # all_flat_indices rng = np.random.default_rng(seed_random)

    all_flat_indices = []

    for c in obs_channels:
        if periodic:
            # Use signed-style offsets, but avoid duplicates modulo Nx.
            # These represent all Nx periodic diagonals.
            candidate_offsets = np.arange(Nx)

            selected_offsets = rng.choice(
                candidate_offsets,
                size=n_diagonals,
                replace=False,
            )

            channel_indices = []
            for d in selected_offsets:
                for i in range(Nx):
                    j = (i + d) % Nx
                    flat_idx = c * Nx * Nx + i * Nx + j
                    channel_indices.append(flat_idx)

        else:
            # Classical matrix diagonals:
            # d = 0 main diagonal
            # d > 0 superdiagonal
            # d < 0 subdiagonal
            candidate_offsets = np.arange(-(Nx - 1), Nx)

            selected_offsets = rng.choice(
                candidate_offsets,
                size=n_diagonals,
                replace=False,
            )

            channel_indices = []
            for d in selected_offsets:
                for i in range(Nx):
                    j = i + d
                    if 0 <= j < Nx:
                        flat_idx = c * Nx * Nx + i * Nx + j
                        channel_indices.append(flat_idx)

        channel_indices = np.array(channel_indices, dtype=np.int64)

        # Remove fixed linear indices if needed.
        if fixed_by_channel is not None and c in fixed_by_channel:
            fixed_idx = np.asarray(fixed_by_channel[c], dtype=np.int64)
            channel_indices = np.setdiff1d(
                channel_indices,
                fixed_idx,
                assume_unique=False,
            )

        all_flat_indices.append(channel_indices)

    all_flat_indices = np.concatenate(all_flat_indices)
    all_flat_indices = np.unique(all_flat_indices)

    return torch.tensor(all_flat_indices, dtype=torch.long, device=device)


def make_random_nonlinear_indices_by_obs_mode(
    obs_mode, Nx, obs_channels, fixed_by_channel,
    cycle, seed_obs, device="cpu", obs_percent_random=None,
    obs_min=None, obs_max=None, rng_obs_percent=None,
    k_nonoverlap=8, n_blocks=10, block_shape=(4, 4),
    n_diagonals=10, diagonal_periodic=True, seed_nonoverlap=0):
    """
    Dispatcher for the random nonlinear observation indices.

    Supported obs_mode values:
      - "random_percent": current fixed-linear + random-atan setting.
      - "kstep_nonoverlap": random nonlinear sites are nonoverlapping over k steps.
      - "block": random nonlinear sites come from disjoint M by N rectangles.
      - "diagonal": random nonlinear sites are on main/sub/super diagonals.
    """
    obs_mode = str(obs_mode).lower()

    if obs_mode == "random_percent":
        if rng_obs_percent is None or obs_min is None or obs_max is None:
            raise ValueError("random_percent mode requires rng_obs_percent, obs_min, and obs_max.")
        obs_percent_random_DA = float(rng_obs_percent.uniform(obs_min, obs_max))
        idx_random = random_obs_indices_excluding_fixed_equal_channels(
            Nx=Nx,
            obs_percent_random=obs_percent_random_DA,
            obs_channels=obs_channels,
            fixed_by_channel=fixed_by_channel,
            seed_random=seed_obs,
            device=device,
        )

    elif obs_mode == "kstep_nonoverlap":
        if obs_percent_random is None:
            raise ValueError("kstep_nonoverlap mode requires obs_percent_random.")
        obs_percent_random_DA = float(obs_percent_random)
        idx_random = kstep_nonoverlap_obs_indices_equal_channels(
            Nx=Nx,
            obs_percent_random=obs_percent_random,
            obs_max=obs_max,
            obs_channels=obs_channels,
            fixed_by_channel=fixed_by_channel,
            cycle=cycle,
            k_nonoverlap=k_nonoverlap,
            base_seed=seed_nonoverlap,
            device=device,
        )

    elif obs_mode == "block":
        idx_random = block_rect_obs_indices_equal_channels(
            Nx=Nx,
            obs_channels=obs_channels,
            fixed_by_channel=fixed_by_channel,
            n_blocks=n_blocks,
            block_shape=block_shape,
            seed_random=seed_obs,
            device=device)
        obs_percent_random_DA = 100.0 * idx_random.numel() / (len(obs_channels) * Nx * Nx)

    elif obs_mode == "diagonal":
        idx_random = random_k_diagonal_obs_indices_equal_channels(
            Nx=Nx, n_diagonals=n_diagonals, obs_channels=obs_channels,
            seed_random=seed_obs, device=device, periodic=diagonal_periodic,
            fixed_by_channel=fixed_by_channel)
        
        obs_percent_random_DA = 100.0 * idx_random.numel() / (len(obs_channels) * Nx * Nx)

    else:
        raise ValueError(
            f"Unknown OBS_MODE={obs_mode!r}. Use one of "
            "'random_percent', 'kstep_nonoverlap', 'block', or 'diagonal'."
        )

    return idx_random, obs_percent_random_DA



def obs_forward_mixed(x, obs_mask_fixed, obs_mask_random, phi_nonlin):
    """
    Mixed observation operator:
      fixed sites  : direct/linear observations
      random sites : nonlinear observations phi_nonlin(x)

    x: (C,H,W) or (Ne,C,H,W)
    returns: (n_obs,) or (Ne,n_obs), ordered as [fixed linear, random nonlinear].
    """
    x_fixed = select_obs(x, obs_mask_fixed)
    x_random = select_obs(x, obs_mask_random)

    if x.dim() == 3:
        return torch.cat([x_fixed, phi_nonlin(x_random)], dim=0)
    if x.dim() == 4:
        return torch.cat([x_fixed, phi_nonlin(x_random)], dim=1)
    raise ValueError(f"obs_forward_mixed expects 3D or 4D tensor, got shape {tuple(x.shape)}")


def observe_norm_mixed(x_norm, obs_mask_fixed, obs_mask_random, obs_sigma, phi_nonlin, generator=None):
    """
    y = [x_norm[fixed_mask], phi_nonlin(x_norm[random_mask])] + noise.
    x_norm: (C,H,W) -> y: (n_fixed+n_random,)
    """
    y = obs_forward_mixed(x_norm, obs_mask_fixed, obs_mask_random, phi_nonlin).to(torch.float32)
    if obs_sigma > 0:
        y = y + obs_sigma * torch.randn(y.shape, device=y.device, generator=generator)
    return y


def observation_rmse_mixed(x_norm, y_obs, obs_mask_fixed, obs_mask_random, phi_nonlin):
    pred = obs_forward_mixed(x_norm, obs_mask_fixed, obs_mask_random, phi_nonlin).cpu()
    return torch.sqrt(torch.mean((pred - y_obs.reshape(-1).cpu()) ** 2)).item()
    
def random_obs_mask(n_channels, nx, obs_count, obs_channels, seed=0, device="cpu"):
    """Random sparse mask: obs_count True sites on obs_channels. Returns (C, H, W) bool."""
    rng = np.random.default_rng(seed)
    mask = torch.zeros(n_channels, nx, nx, dtype=torch.bool)
    candidates = [(int(c), i, j) for c in obs_channels for i in range(nx) for j in range(nx)]
    pick = rng.choice(len(candidates), size=obs_count, replace=False)
    for k in pick:
        c, i, j = candidates[k]
        mask[c, i, j] = True
    return mask.to(device)


def make_obs_phi_atan(alpha):
    def phi(u):
        return torch.atan(alpha * u)
    return phi


def make_obs_phi_prime_atan(alpha):
    def phi_prime(u):
        au = alpha * u
        return alpha / (1.0 + au.pow(2))
    return phi_prime

def obs_phi_direct(u):
    return u

def obs_phi_prime_direct(u):
    return torch.ones_like(u)


def is_full_observation(obs_mask):
    return bool(obs_mask.all().item())


def has_unobserved_sites(obs_mask):
    return bool((~obs_mask).any().item())


def obs_forward(x, obs_mask, phi):
    """Apply phi at observed sites. x: (C,H,W) or (Ne,C,H,W) -> (n_obs,) or (Ne,n_obs)."""
    if is_full_observation(obs_mask):
        if x.dim() == 3:
            return phi(x.reshape(-1))
        if x.dim() == 4:
            return phi(x.reshape(x.shape[0], -1))
    return phi(select_obs(x, obs_mask))


def select_obs(x, obs_mask):
    """Select observed sites. x: (C,H,W) -> (n_obs,); (Ne,C,H,W) -> (Ne,n_obs)."""
    m = obs_mask.to(x.device)
    if x.dim() == 3:
        return x[m]
    if x.dim() == 4:
        return x[:, m]
    raise ValueError(f"select_obs expects 3D or 4D tensor, got shape {tuple(x.shape)}")


def observe_norm(x_norm, obs_mask, obs_sigma, phi, generator=None):
    """y = phi(x_norm[obs_mask]) + noise. x_norm: (C,H,W) -> y: (n_obs,)"""
    y = obs_forward(x_norm, obs_mask, phi).to(torch.float32)
    if obs_sigma > 0:
        y = y + obs_sigma * torch.randn(y.shape, device=y.device, generator=generator)
    return y


def observation_rmse(x_norm, y_obs, obs_mask, phi):
    pred = obs_forward(x_norm, obs_mask, phi).cpu()
    return torch.sqrt(torch.mean((pred - y_obs.reshape(-1).cpu()) ** 2)).item()

### Elementwise observation operator

Uncomment **one** block: **atan** $$g(x)=\arctan(\alpha x)$$ or **direct** $$g(x)=x.$$


In [ ]:
# --- Elementwise observation operator ---
obs_sigma = 0.025  # noise std in observation units

# --- atan: y = atan(alpha * x[mask]) + noise ---
obs_kind = "atan"
OBS_ALPHA = 13.0
obs_phi = make_obs_phi_atan(OBS_ALPHA)
obs_phi_prime = make_obs_phi_prime_atan(OBS_ALPHA)

# # --- direct: y = x[mask] + noise ---
# obs_kind = "direct"
# OBS_ALPHA = 1.0
# obs_phi = obs_phi_direct
# obs_phi_prime = obs_phi_prime_direct


### EDM sampling with relaxed residual normalization and localized weighted $J_F$

For each ensemble member, the mixed observation likelihood is evaluated at the denoised state $\hat x_0$,

$$
\mathcal L_e
=
\frac{1}{2\sigma_{\mathrm{obs}}^2}
\left\|g(\hat x_{0,e})-y\right\|_2^2.
$$

The denoised-state likelihood gradient is split into linear and nonlinear contributions,

$$
\widetilde g_{\rm lin}
=
\lambda_{\rm lin}g_{\rm lin},
\qquad
\widetilde g_{\rm nonlin}
=
\lambda_{\rm nonlin}g_{\rm nonlin}.
$$

For the EDM denoiser,

$$
\hat x_0
=
c_{\rm skip}x
+
c_{\rm out}F(c_{\rm in}x,H),
$$

so, with the history condition $H$ held fixed,

$$
J_D
=
c_{\rm skip}I
+
c_{\rm out}c_{\rm in}J_F.
$$

The guidance action used by the sampler is therefore approximated by

$$
A_e
=
c_{\rm skip}
\left(
\widetilde g_{{\rm lin},e}
+
\widetilde g_{{\rm nonlin},e}
\right)
+
c_{\rm out}c_{\rm in}
\left[
\alpha_{F,\rm lin}J_F^T\widetilde g_{{\rm lin},e}
+
\alpha_{F,\rm nonlin}J_F^T\widetilde g_{{\rm nonlin},e}
\right].
$$

The $J_F$ action is estimated by localized weighted ridge regression. For an observed output at layer $c$ and spatial location $(i,j)$, the local input neighborhood is the periodic square

$$
\mathcal N_{ij}
=
\{(c',i+\Delta i,j+\Delta j):
c'=1,2,\;
|\Delta i|,|\Delta j|\le R\}.
$$

Thus the local predictor dimension is

$$
d_{\rm loc}=2(2R+1)^2.
$$

With `NE=36`, the default `jf_local_radius=1` gives $d_{\rm loc}=18$. Radius 2 would give $50$ predictors and is therefore not the default.

The relaxed residual normalization is retained,

$$
\rho_e
=
\sqrt{\|r_e\|_2^2+c^2},
\qquad
c^2
=
\alpha_{\rm soft}^2m\sigma_{\rm obs}^2.
$$

The SQG sampler keeps its existing additive guidance placement and uses

$$
x\leftarrow x+\frac{A_e}{\rho_e}.
$$

This changes the Jacobian approximation without simultaneously changing the reverse-sampler discretization.

**Caveat:** the ensemble members generally have different history conditions. The local regression is therefore the same empirical ensemble approximation used in the L96 version; it is not a strict fixed-history derivative unless the histories are held fixed across the regression ensemble.


In [ ]:
NUM_STEPS = 20

In [ ]:
# --- EDM sampling with relaxed residual normalization + localized weighted J_F ---
def _edm_sigma_data(net, fallback=0.5):
    value = getattr(net, "sigma_data", fallback)
    try:
        return float(torch.as_tensor(value).detach().cpu().reshape(-1)[0].item())
    except Exception:
        return float(value)


def _c_skip(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sd.pow(2) / (sigma.pow(2) + sd.pow(2))


def _c_in(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return 1.0 / torch.sqrt(sigma.pow(2) + sd.pow(2))


def _c_out(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sigma * sd / torch.sqrt(sigma.pow(2) + sd.pow(2))


def _c_out_c_in(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sigma * sd / (sigma.pow(2) + sd.pow(2))


def _local_weighted_fbar_jf_transpose_actions_2d(
    x_noisy, x0_denoised, g_list, t_stage, sigma_data, center_idx,
    radius=1, bandwidth_factor=1.0, ridge_rel=1e-6, ridge_abs=1e-10, eps=1e-12
):
    """Localized weighted approximation of J_F^T g for periodic 2D SQG fields.

    x_noisy, x0_denoised: (Ne, C, H, W)
    center_idx: flattened output indices where likelihood gradients can be nonzero.

    For each observed output center j=(c,i,k), fit

        F_j - Fbar_j ~= b_j^T (U_N(j) - Ubar_N(j)),

    where U=c_in*x and N(j) is a periodic square spatial patch containing all
    input channels. The fitted b_j is a local row approximation of J_F.

    The transpose action is assembled by accumulating g_j*b_j onto overlapping
    local input patches. The same local regressions are reused for every g in g_list.
    """
    device, dtype = x_noisy.device, x_noisy.dtype
    if x_noisy.ndim != 4:
        raise ValueError(f"SQG localized regression expects (Ne,C,H,W), got {tuple(x_noisy.shape)}")
    if x0_denoised.shape != x_noisy.shape:
        raise ValueError("x0_denoised must have the same shape as x_noisy.")

    ne, n_channels, height, width = x_noisy.shape
    radius = int(radius)
    if radius < 0:
        raise ValueError("jf_local_radius must be nonnegative.")

    centers = torch.unique(center_idx.to(device=device, dtype=torch.long), sorted=True)
    if centers.numel() == 0:
        return [torch.zeros_like(g) for g in g_list]

    total_dim = n_channels * height * width
    if int(centers.min()) < 0 or int(centers.max()) >= total_dim:
        raise ValueError("center_idx contains a flattened SQG index outside the state dimension.")

    spatial_size = height * width
    center_spatial = centers % spatial_size
    center_row = center_spatial // width
    center_col = center_spatial % width

    offsets = torch.arange(-radius, radius + 1, device=device, dtype=torch.long)
    dy, dx = torch.meshgrid(offsets, offsets, indexing="ij")
    dy = dy.reshape(-1)
    dx = dx.reshape(-1)

    patch_row = (center_row[:, None] + dy[None, :]) % height
    patch_col = (center_col[:, None] + dx[None, :]) % width
    patch_spatial = patch_row * width + patch_col  # (n_centers, patch_area)

    channel_offsets = (
        torch.arange(n_channels, device=device, dtype=torch.long) * spatial_size
    )
    patch_idx = (
        channel_offsets[None, :, None] + patch_spatial[:, None, :]
    ).reshape(centers.numel(), -1)
    local_dim = patch_idx.shape[1]

    cskip = _c_skip(t_stage, sigma_data, device, dtype)
    cin = _c_in(t_stage, sigma_data, device, dtype)
    cout = _c_out(t_stage, sigma_data, device, dtype)

    u = (cin * x_noisy).reshape(ne, total_dim)
    cout_safe = torch.clamp(cout, min=torch.as_tensor(eps, device=device, dtype=dtype))
    f = ((x0_denoised - cskip * x_noisy) / cout_safe).reshape(ne, total_dim)

    # local U: (n_centers, Ne, local_dim), center F: (n_centers, Ne)
    u_local = u[:, patch_idx].permute(1, 0, 2).contiguous()
    f_center = f[:, centers].transpose(0, 1).contiguous()

    du = u_local - u_local.mean(dim=1, keepdim=True)
    df = f_center - f_center.mean(dim=1, keepdim=True)

    distances = torch.linalg.vector_norm(du, ord=2, dim=2)
    median_distance = torch.median(distances, dim=1).values
    rms_distance = torch.sqrt(torch.mean(distances.pow(2), dim=1)).clamp_min(eps)
    median_distance = torch.where(median_distance > eps, median_distance, rms_distance)

    h = (
        torch.as_tensor(float(bandwidth_factor), device=device, dtype=dtype) * median_distance
    ).clamp_min(eps)
    weights = torch.exp(-0.5 * (distances / h[:, None]).pow(2))
    weights = weights / weights.mean(dim=1, keepdim=True).clamp_min(eps)
    sqrt_w = torch.sqrt(torch.clamp(weights, min=0.0))

    a = sqrt_w[:, :, None] * du
    c = sqrt_w * df

    gram = torch.einsum("snp,snq->spq", a, a)
    rhs = torch.einsum("snp,sn->sp", a, c)

    gram_scale = gram.diagonal(dim1=-2, dim2=-1).abs().sum(dim=1) / max(local_dim, 1)
    ridge = (
        torch.as_tensor(float(ridge_abs), device=device, dtype=dtype)
        + torch.as_tensor(float(ridge_rel), device=device, dtype=dtype) * gram_scale
    )
    eye = torch.eye(local_dim, device=device, dtype=dtype).unsqueeze(0)
    system = gram + ridge[:, None, None] * eye

    try:
        b = torch.linalg.solve(system, rhs.unsqueeze(-1)).squeeze(-1)
    except RuntimeError:
        b = (torch.linalg.pinv(system) @ rhs.unsqueeze(-1)).squeeze(-1)
    if not torch.isfinite(b).all():
        b = (torch.linalg.pinv(system) @ rhs.unsqueeze(-1)).squeeze(-1)

    # Accumulate overlapping local transpose contributions.
    scatter_idx = patch_idx.reshape(1, -1).expand(ne, -1)
    outputs = []
    for g in g_list:
        if g.shape != x_noisy.shape:
            raise ValueError("Each guidance tensor in g_list must have shape (Ne,C,H,W).")
        gf = g.reshape(ne, total_dim)
        g_center = gf[:, centers].transpose(0, 1).contiguous()
        contribution = (
            g_center[:, :, None] * b[:, None, :]
        ).permute(1, 0, 2).reshape(ne, -1)

        out = torch.zeros((ne, total_dim), device=device, dtype=dtype)
        out.scatter_add_(1, scatter_idx, contribution)
        outputs.append(out.reshape_as(g))

    return outputs


def edm_sampler_obs(
    net,
    latents,      # (Ne, C, H, W) initial noise; scaled to x at sigma_max
    conditions,   # (Ne, n_cond, H, W) normalized condition stack
    obs=None, class_labels=None, randn_like=torch.randn_like, num_steps=18,
    sigma_min=0.002, sigma_max=80,
    rho=7, S_churn=0, S_min=0, S_max=float("inf"), S_noise=1):

    """Conditional EDM (Heun) + relaxed residual norm + localized weighted J_F.

    Denoiser Jacobian action:
        J_D^T g ~= c_skip*g + c_out*c_in*J_F^T*g

    The linear/nonlinear likelihood contributions have separate lambda and alpha_F
    factors. The SQG sampler retains its existing additive post-Heun guidance update.
    """
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    use_obs = obs is not None and obs.get("y_obs") is not None
    if use_obs:
        device = latents.device
        obs_mask_fixed = obs["obs_mask_fixed"].to(device)
        obs_mask_random = obs["obs_mask_random"].to(device)
        obs_mask_all = obs_mask_fixed | obs_mask_random
        center_idx = torch.nonzero(obs_mask_all.reshape(-1), as_tuple=False).reshape(-1)

        obs_sigma = float(obs["obs_sigma"])
        residual_soften_alpha = float(obs.get("residual_soften_alpha", 1.0))
        guidance_scale_lin = float(obs.get("guidance_scale_lin", 1.0))
        guidance_scale_nonlin = float(obs.get("guidance_scale_nonlin", 1.0))
        jf_scale_lin = float(obs.get("jf_scale_lin", 1.0))
        jf_scale_nonlin = float(obs.get("jf_scale_nonlin", 1.0))
        jf_local_radius = int(obs.get("jf_local_radius", 1))
        jf_weight_bandwidth_factor = float(obs.get("jf_weight_bandwidth_factor", 1.0))
        jf_ridge_rel = float(obs.get("jf_ridge_rel", 1e-6))
        jf_ridge_abs = float(obs.get("jf_ridge_abs", 1e-10))
        edm_sigma_data = _edm_sigma_data(
            net, fallback=float(obs.get("edm_sigma_data_fallback", 0.5))
        )
        phi_nonlin = obs["phi_nonlin"]

        y_obs = obs["y_obs"].to(device, torch.float64)
        if y_obs.dim() == 1:
            y_obs = y_obs.unsqueeze(0)

    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])])

    x_next = latents.to(torch.float64) * t_steps[0]
    neg_log_likelihoods = [] if use_obs else None
    residual_norm_history = [] if use_obs else None

    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat**2 - t_cur**2).sqrt() * S_noise * randn_like(x_cur)

        jac_action = None
        rho_member = None
        x0_norm_pred = None

        if use_obs:
            with torch.no_grad():
                x0_norm = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)

            # Raw denoised-state likelihood gradient.
            x0_leaf = x0_norm.detach().requires_grad_(True)
            x_fixed = x0_leaf[:, obs_mask_fixed]
            x_random = phi_nonlin(x0_leaf[:, obs_mask_random])
            gx = torch.cat([x_fixed, x_random], dim=1)
            resid = gx - y_obs

            n_obs = resid.shape[1]
            soften_c2 = residual_soften_alpha**2 * n_obs * obs_sigma**2
            rho_member = torch.sqrt(
                torch.sum(resid.pow(2), dim=1) + soften_c2
            ).clamp_min(1e-12).detach()

            nll = 0.5 * (resid.pow(2) / obs_sigma**2).sum(dim=1)
            log_p = -nll.sum()
            raw_guidance = torch.autograd.grad(log_p, x0_leaf)[0].detach()

            # Lambda_L scales apply to both direct and J_F branches.
            fixed_mask_view = obs_mask_fixed.unsqueeze(0).to(dtype=raw_guidance.dtype)
            random_mask_view = obs_mask_random.unsqueeze(0).to(dtype=raw_guidance.dtype)
            g_lin = guidance_scale_lin * raw_guidance * fixed_mask_view
            g_nonlin = guidance_scale_nonlin * raw_guidance * random_mask_view

            # Local weighted ensemble approximation to J_F^T g.
            jf_lin, jf_nonlin = _local_weighted_fbar_jf_transpose_actions_2d(
                x_hat,
                x0_norm,
                [g_lin, g_nonlin],
                t_hat,
                edm_sigma_data,
                center_idx,
                radius=jf_local_radius,
                bandwidth_factor=jf_weight_bandwidth_factor,
                ridge_rel=jf_ridge_rel,
                ridge_abs=jf_ridge_abs,
            )

            cskip = _c_skip(t_hat, edm_sigma_data, x_hat.device, x_hat.dtype)
            coutcin = _c_out_c_in(t_hat, edm_sigma_data, x_hat.device, x_hat.dtype)

            direct = g_lin + g_nonlin
            jf_action = jf_scale_lin * jf_lin + jf_scale_nonlin * jf_nonlin
            jac_action = cskip * direct + coutcin * jf_action

            neg_log_likelihoods.append(float(nll.mean().detach().cpu()))
            residual_norm_history.append(float(rho_member.mean().detach().cpu()))
            x0_norm_pred = x0_norm

        with torch.no_grad():
            if x0_norm_pred is None:
                x0_norm_pred = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)

            d_cur = (x_hat - x0_norm_pred) / t_hat
            x_next = x_hat + (t_next - t_hat) * d_cur

            if i < num_steps - 1:
                denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
                d_prime = (x_next - denoised) / t_next
                x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

            if use_obs:
                rho_view = rho_member.reshape(
                    [rho_member.shape[0]] + [1] * (jac_action.ndim - 1)
                )
                x_next = x_next + jac_action / rho_view

    if use_obs:
        return x_next, neg_log_likelihoods, residual_norm_history
    return x_next


def edm_sigma_schedule(num_steps, net, sigma_min=0.002, sigma_max=80, rho=7, device=None):
    """EDM noise levels sigma(t_hat) at each sampler step."""
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)
    dev = device if device is not None else next(net.parameters()).device
    step_indices = torch.arange(num_steps, dtype=torch.float64, device=dev)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros(1, dtype=torch.float64, device=dev)])
    return t_steps[:-1]


## Phase 4: DA loop (re-runnable, normalized space)

Each cycle runs **analysis** (EDM + obs guidance, condition on prior analysis).

Optional comparisons (`RUN_EXTRA_COMPARISONS`):

- **no-DA** — pure EDM forecast, never uses observations, own history
- **bgrd** — traditional background: condition on prior **analysis**, no current obs


### Phase 4a: Forecast only and DA

In [ ]:
# NUM_STEPS is configured in the EDM-sampling section above.


In [ ]:
# --- DA run settings (edit and re-run this cell) ---
SAVE_RESULTS = False
SAVE_PATH = os.path.join(trajectory_dir, "ensf_atan.pt")
RUN_EXTRA_COMPARISONS = True  # no-DA + traditional background (bgrd) forecast

In [ ]:
# --- Reset state ---
core_kw = params.to_core_kwargs()
truth = SQGSolver(path_burn[-1].to(device), **core_kw)
truth.set_state(state_after_burn)

n_burn = path_burn.shape[0]  # T_burn+1
burn_norm = [normalize_tensor(path_burn[i], norm_stats) for i in range(n_burn)]
truth_frames = [path_burn[i].clone() for i in range(n_burn)]  # physical (C, H, W)
truth_norm_frames = [burn_norm[i].clone() for i in range(n_burn)]  # normalized (C, H, W)

In [ ]:
hist_noda = [] if RUN_EXTRA_COMPARISONS else None

In [ ]:
for k in range(NE):
    # hf = [path_burn[i].clone() for i in range(n_burn)]
    hf = [burn_norm[i].clone() for i in range(n_burn)]
    hf[-1] = perturb_norm(hf[-1], ENSEMBLE_PERTURB_STD, SEED + 1000 + k)
    if RUN_EXTRA_COMPARISONS:
        hist_noda.append([f.clone() for f in hf])

In [ ]:
rnd_forecast = StackedRandomGenerator(device, [SEED_Forecast + k for k in range(NE)])
H, W = NX, NX

In [ ]:
rmse_truth_vs_noda = []
spread_noda = []

In [ ]:
cycle_times = []
pbar = tqdm.tqdm(range(N_DA_CYCLES), desc="DA cycles", unit="cycle")
for cycle in pbar:
    t_cycle_start = time.perf_counter()

    truth.step(timesteps_per_output)
    pv_truth = truth.fields()["pv"].detach().cpu()
    truth_frames.append(pv_truth.clone())

    pv_truth_norm = normalize_tensor(pv_truth, norm_stats)
    truth_norm_frames.append(pv_truth_norm.clone())

    if RUN_EXTRA_COMPARISONS:
        cond_noda = build_condition_from_history_batch(hist_noda, HISTORY).to(device)
        if not torch.isfinite(cond_noda).all():
            print(f"Stopping at cycle {cycle}: non-finite DM conditions (no-DA history)")
            break

        lat_n = rnd_forecast.randn([NE, n_target, H, W], device=device)
        with torch.no_grad():
            sample_noda_norm = edm_sampler_obs(
                net, lat_n, cond_noda, num_steps=NUM_STEPS,
            ).to(torch.float32)

        next_noda = sample_noda_norm.cpu()
        if not torch.isfinite(next_noda).all():
            print(f"Stopping at cycle {cycle}: non-finite DM ensemble sample")
            break

        rmse_n, spr_n = ensemble_batch_rmse(next_noda, pv_truth_norm)
        if not all(np.isfinite([rmse_n, spr_n])):
            print(f"Stopping at cycle {cycle}: non-finite RMSE/spread metrics")
            break

        for k in range(NE):
            hist_noda[k].append(next_noda[k].clone())

        rmse_truth_vs_noda.append(rmse_n)
        spread_noda.append(spr_n)

    cycle_times.append(time.perf_counter() - t_cycle_start)
    avg_step = sum(cycle_times) / len(cycle_times)
    remaining = avg_step * (N_DA_CYCLES - cycle - 1)
    pbar.set_postfix(step=f"{avg_step:.2f}s", remaining=f"{remaining:.0f}s")


In [ ]:
n_truth_cycles_completed = len(truth_frames) - n_burn
if RUN_EXTRA_COMPARISONS:
    print(
        f"Completed {len(rmse_truth_vs_noda)}/{N_DA_CYCLES} no-DA cycles. "
        f"Last RMSE={rmse_truth_vs_noda[-1] if rmse_truth_vs_noda else float('nan'):.4f}"
    )
else:
    print(f"Completed {n_truth_cycles_completed}/{N_DA_CYCLES} truth-only cycles.")

if cycle_times:
    print(f"Avg time per cycle: {sum(cycle_times) / len(cycle_times):.2f}s")


In [ ]:
# --- Result containers (normalized ens; truth physical) ---
path_truth_full = torch.stack(truth_frames, dim=0)
path_truth_norm_full = torch.stack(truth_norm_frames, dim=0)

path_noda_ens = (
    torch.stack([torch.stack(h, dim=0) for h in hist_noda], dim=0)
    if RUN_EXTRA_COMPARISONS
    else None
)
path_noda = path_noda_ens.mean(dim=0) if path_noda_ens is not None else None

In [ ]:
from scipy.io import savemat

# Move tensors to CPU numpy arrays
path_truth_np = path_truth_full.detach().cpu().numpy()
# path_truth_norm_np = path_truth_norm_full.detach().cpu().numpy()

In [ ]:
path_noda_np = (
    denormalize_tensor(path_noda_ens, norm_stats).detach().cpu().numpy()
    if path_noda_ens is not None
    else None
)


In [ ]:
rmse_noda = torch.tensor(rmse_truth_vs_noda, dtype = torch.float32)
spread_noda = torch.tensor(spread_noda, dtype = torch.float32)

In [ ]:
rmse_noda_np = rmse_noda.detach().cpu().numpy()
spread_noda_np = spread_noda.detach().cpu().numpy()

In [ ]:
import os

In [ ]:
folder1 = "../../output_DA/Figures/SQG_DA_1h/MixedObs"
os.makedirs(folder1, exist_ok=True)

filename = (
    f"Truth_and_Forecast_Ncycles{N_DA_CYCLES}"
    f"_Tburnin{T_BURN_IN}"
    f"_NE{NE}"
    f"_EnSF_MixedObs.mat"
)

save_path = os.path.join(folder1, filename)

In [ ]:
forecast_save_dict = {
    "path_truth_full": path_truth_np,
    "t_burn_in": T_BURN_IN,
    "Ncycle": N_DA_CYCLES,
    "Nen": NE,
}
if RUN_EXTRA_COMPARISONS and path_noda_np is not None:
    forecast_save_dict.update({
        "path_noda": path_noda_np,
        "rmse_noda": rmse_noda_np,
        "spread_noda": spread_noda_np,
    })

savemat(save_path, forecast_save_dict)


### Phase 4b: Analysis

### Observation mask

Uncomment **one** block. Each sets ``obs_mask`` with shape ``(2, NX, NX)`` (bool).

**Full** observation (all True) is supported; unobserved-site metrics are skipped automatically.


In [ ]:
# if "obs_mask" in globals():
#     check_obs_mask(obs_mask, n_target, NX)
# else:
#     print("Skipping demo mask check because obs_mask has not been constructed yet.")


In [ ]:
H, W = NX, NX

In [ ]:
SEED_OBS_PERCENT = 111222

In [ ]:
# Observation operator is configured once in the earlier "Elementwise observation operator" cell.
print(f"obs_kind={obs_kind}, alpha={OBS_ALPHA}, obs_sigma={obs_sigma}")


In [ ]:
s_churn = 0.0
s_min = 0.01
s_max = 10.0

In [ ]:
sigma_max = 80.0
sigma_min = 0.002

In [ ]:
obs_channels = [0, 1]

In [ ]:
# Likelihood-gradient scales.
GUIDANCE_SCALE_LIN = 5.0e-05
GUIDANCE_SCALE_NONLIN = 1.3e-04

# Localized J_F scales and regression settings.
JF_SCALE_LIN = 1.0
JF_SCALE_NONLIN = 1.0
JF_LOCAL_RADIUS = 1
JF_WEIGHT_BANDWIDTH_FACTOR = 1.0
JF_RIDGE_REL = 1e-6
JF_RIDGE_ABS = 1e-10
EDM_SIGMA_DATA_FALLBACK = 0.5

# Relaxed residual normalization.
RESIDUAL_SOFTEN_ALPHA = 1.0

latent_inflation = 1.0
posterior_gamma = 1.0


In [ ]:
# ----------------------------------------------------------------------
# Observation modes and reproducible structural settings
# ----------------------------------------------------------------------
obs_min = 10.0
obs_max = 20.0
obs_channels = [0, 1]

SEED_OBS_INDEX = 23456
SEED_OBS_PERCENT = 111222
SEED_OBS_NOISE = SEED + 200000
SEED_DIAGONAL = 0
SEED_FIXED = 12345

OBS_MODE_CONFIGS = {
    "random_percent": {
        "obs_percent_fix": 5.0,
        "obs_min": 10.0,
        "obs_max": 20.0,
    },
    "kstep_nonoverlap": {
        "obs_percent_fix": 5.0,
        "obs_min": 10.0,
        "obs_max": 20.0,
        "k_nonoverlap": 3,
    },
    "block": {
        "obs_percent_fix": 0.0,
        "n_blocks": 10,
        "block_shape": (10, 10),
    },
    "diagonal": {
        "obs_percent_fix": 0.0,
        "n_diagonals_min": 10,
        "n_diagonals_max": 16,
        "diagonal_periodic": True,
    },
}

OBS_MODE = "random_percent"
# OBS_MODE = "kstep_nonoverlap"
# OBS_MODE = "block"
# OBS_MODE = "diagonal"

if OBS_MODE not in OBS_MODE_CONFIGS:
    raise ValueError(f"Unknown OBS_MODE={OBS_MODE!r}; choose from {tuple(OBS_MODE_CONFIGS)}.")

# Backward-compatible structural names used by later filename cells.
K_NONOVERLAP = OBS_MODE_CONFIGS["kstep_nonoverlap"]["k_nonoverlap"]
N_BLOCKS = OBS_MODE_CONFIGS["block"]["n_blocks"]
BLOCK_SHAPE = OBS_MODE_CONFIGS["block"]["block_shape"]
DIAGONAL_PERIODIC = OBS_MODE_CONFIGS["diagonal"]["diagonal_periodic"]


In [ ]:
def scale_ensemble_spread(X, gamma):
    mean = X.mean(dim=0, keepdim=True)
    return mean + gamma * (X - mean)


def make_parameter_grid(grid):
    """Cartesian product of a compact parameter grid -> list of override dicts."""
    keys = list(grid)
    return [dict(zip(keys, values)) for values in product(*(grid[k] for k in keys))]


def run_da_configuration(
    obs_mode,
    overrides=None,
    keep_history=False,
    show_progress=False,
    n_cycles=None,
    desc=None,
):
    """Run one SQG DA configuration from the same post-burn-in state.

    Every call resets:
      - truth solver to state_after_burn,
      - ensemble histories to the same perturbed burn-in states,
      - observation-index/coverage RNGs,
      - observation noise by cycle,
      - diffusion latent RNGs.

    Therefore sweep candidates are directly comparable.
    """
    if obs_mode not in OBS_MODE_CONFIGS:
        raise ValueError(f"Unknown obs_mode={obs_mode!r}")

    cfg_run = {
        **OBS_MODE_CONFIGS[obs_mode],
        "guidance_scale_lin": GUIDANCE_SCALE_LIN,
        "guidance_scale_nonlin": GUIDANCE_SCALE_NONLIN,
        "jf_scale_lin": JF_SCALE_LIN,
        "jf_scale_nonlin": JF_SCALE_NONLIN,
        "jf_local_radius": JF_LOCAL_RADIUS,
        "jf_weight_bandwidth_factor": JF_WEIGHT_BANDWIDTH_FACTOR,
        "jf_ridge_rel": JF_RIDGE_REL,
        "jf_ridge_abs": JF_RIDGE_ABS,
        "edm_sigma_data_fallback": EDM_SIGMA_DATA_FALLBACK,
        "residual_soften_alpha": RESIDUAL_SOFTEN_ALPHA,
        "latent_inflation": latent_inflation,
        "posterior_gamma": posterior_gamma,
    }
    if overrides:
        cfg_run.update(overrides)

    n_cycles = N_DA_CYCLES if n_cycles is None else int(n_cycles)
    if n_cycles < 1:
        raise ValueError("n_cycles must be >= 1")

    # Reset truth.
    truth_local = SQGSolver(path_burn[-1].to(device), **core_kw)
    truth_local.set_state(state_after_burn)

    # Reset analysis ensemble histories.
    hist_local = []
    for k in range(NE):
        ha = [f.clone() for f in burn_norm]
        ha[-1] = perturb_norm(ha[-1], ENSEMBLE_PERTURB_STD, SEED + 1000 + k)
        hist_local.append([f.clone() for f in ha])

    # Reset all RNG streams so every sweep candidate sees the same experiment.
    rng_obs_local = np.random.default_rng(SEED_OBS_INDEX)
    rng_obs_percent_local = np.random.default_rng(SEED_OBS_PERCENT)
    rng_diagonal_local = np.random.default_rng(SEED_DIAGONAL)
    rnd_local = StackedRandomGenerator(device, [int(SEED_DM) + k for k in range(NE)])

    obs_percent_fix_local = float(cfg_run.get("obs_percent_fix", 0.0))
    fixed_by_channel_local = make_fixed_obs_indices_equal_channels(
        Nx=NX,
        obs_percent_fix=obs_percent_fix_local,
        obs_channels=obs_channels,
        seed_fix=SEED_FIXED,
    )
    if obs_mode not in ("block", "diagonal"):
        obs_flat_idx_fixed_local = fixed_obs_indices_equal_channels(
            fixed_by_channel=fixed_by_channel_local,
            obs_channels=obs_channels,
            device=device,
        )
    else:
        obs_flat_idx_fixed_local = torch.empty(0, dtype=torch.long, device=device)

    rmse_local = []
    rmse_obs_local = []
    rmse_unobs_local = []
    spread_local = []
    obs_rmse_local = []
    nll_last_local = []
    rho_last_local = []
    cycle_times_local = []
    obs_pct_local = []
    obs_mask_local = []

    iterator = range(n_cycles)
    if show_progress:
        iterator = tqdm.tqdm(iterator, desc=desc or f"DA [{obs_mode}]", unit="cycle")

    for cycle in iterator:
        t_cycle_start = time.perf_counter()
        seed_obs = int(rng_obs_local.integers(0, 2**32 - 1))

        truth_local.step(timesteps_per_output)
        pv_truth = truth_local.fields()["pv"].detach().cpu()
        pv_truth_norm = normalize_tensor(pv_truth, norm_stats)

        obs_percent_random = None
        if obs_mode in ("random_percent", "kstep_nonoverlap"):
            if obs_mode == "kstep_nonoverlap":
                obs_percent_random = float(
                    rng_obs_percent_local.uniform(cfg_run["obs_min"], cfg_run["obs_max"])
                )

        n_diagonals = int(cfg_run.get("n_diagonals_min", 10))
        if obs_mode == "diagonal":
            n_diagonals = int(
                rng_diagonal_local.integers(
                    int(cfg_run["n_diagonals_min"]),
                    int(cfg_run["n_diagonals_max"]) + 1,
                )
            )

        idx_random, obs_percent_random_DA = make_random_nonlinear_indices_by_obs_mode(
            obs_mode=obs_mode,
            Nx=NX,
            obs_channels=obs_channels,
            fixed_by_channel=fixed_by_channel_local,
            cycle=cycle,
            seed_obs=seed_obs,
            device=device,
            obs_percent_random=obs_percent_random,
            obs_min=cfg_run.get("obs_min"),
            obs_max=cfg_run.get("obs_max"),
            rng_obs_percent=rng_obs_percent_local,
            k_nonoverlap=int(cfg_run.get("k_nonoverlap", 3)),
            n_blocks=int(cfg_run.get("n_blocks", 10)),
            block_shape=tuple(cfg_run.get("block_shape", (10, 10))),
            n_diagonals=n_diagonals,
            diagonal_periodic=bool(cfg_run.get("diagonal_periodic", True)),
            seed_nonoverlap=SEED_OBS_PERCENT,
        )

        obs_mask_fixed = obs_mask_vector(
            obs_flat_idx_fixed_local,
            total_size=2 * NX * NX,
            device=device,
            dtype=torch.bool,
        ).reshape(2, NX, NX)
        obs_mask_random = obs_mask_vector(
            idx_random,
            total_size=2 * NX * NX,
            device=device,
            dtype=torch.bool,
        ).reshape(2, NX, NX)
        obs_mask = obs_mask_fixed | obs_mask_random

        if torch.any(obs_mask_fixed & obs_mask_random):
            raise RuntimeError("Fixed linear and nonlinear observation masks overlap.")

        obs_pct_local.append(float(obs_percent_random_DA))
        if keep_history:
            obs_mask_local.append(obs_mask.detach().cpu().clone())

        obs_noise_gen = torch.Generator(device=pv_truth_norm.device)
        obs_noise_gen.manual_seed(int(SEED_OBS_NOISE + cycle))
        y_obs = observe_norm_mixed(
            pv_truth_norm,
            obs_mask_fixed,
            obs_mask_random,
            obs_sigma,
            obs_phi,
            generator=obs_noise_gen,
        ).to(device)
        if not torch.isfinite(y_obs).all():
            break

        obs_rmse_local.append(
            observation_rmse_mixed(
                pv_truth_norm, y_obs, obs_mask_fixed, obs_mask_random, obs_phi
            )
        )

        cond_analysis = build_condition_from_history_batch(hist_local, HISTORY).to(device)
        if not torch.isfinite(cond_analysis).all():
            break

        obs_cfg = {
            "y_obs": y_obs,
            "obs_mask_fixed": obs_mask_fixed,
            "obs_mask_random": obs_mask_random,
            "obs_sigma": obs_sigma,
            "phi_nonlin": obs_phi,
            "guidance_scale_lin": float(cfg_run["guidance_scale_lin"]),
            "guidance_scale_nonlin": float(cfg_run["guidance_scale_nonlin"]),
            "jf_scale_lin": float(cfg_run["jf_scale_lin"]),
            "jf_scale_nonlin": float(cfg_run["jf_scale_nonlin"]),
            "jf_local_radius": int(cfg_run["jf_local_radius"]),
            "jf_weight_bandwidth_factor": float(cfg_run["jf_weight_bandwidth_factor"]),
            "jf_ridge_rel": float(cfg_run["jf_ridge_rel"]),
            "jf_ridge_abs": float(cfg_run["jf_ridge_abs"]),
            "edm_sigma_data_fallback": float(cfg_run["edm_sigma_data_fallback"]),
            "residual_soften_alpha": float(cfg_run["residual_soften_alpha"]),
        }

        lat_a = float(cfg_run["latent_inflation"]) * rnd_local.randn(
            [NE, n_target, H, W], device=device
        )

        sample_a_norm, nll_hist, rho_hist = edm_sampler_obs(
            net,
            lat_a,
            cond_analysis,
            obs=obs_cfg,
            num_steps=NUM_STEPS,
            sigma_min=sigma_min,
            sigma_max=sigma_max,
            S_churn=s_churn,
            S_min=s_min,
            S_max=s_max,
        )

        gamma_post = float(cfg_run["posterior_gamma"])
        if gamma_post != 1.0:
            sample_a_norm = scale_ensemble_spread(sample_a_norm, gamma_post)

        next_analysis = sample_a_norm.cpu().float()
        if not torch.isfinite(next_analysis).all():
            break

        _, rmse_obs, rmse_unobs = ensemble_rmse_spatial_split(
            next_analysis, pv_truth_norm, obs_mask
        )
        rmse_a, spr_a = ensemble_batch_rmse(next_analysis, pv_truth_norm)
        if not all(np.isfinite([rmse_a, spr_a, rmse_obs])):
            break

        for k in range(NE):
            hist_local[k].append(next_analysis[k].clone())

        rmse_local.append(rmse_a)
        rmse_obs_local.append(rmse_obs)
        rmse_unobs_local.append(rmse_unobs)
        spread_local.append(spr_a)
        nll_last_local.append(nll_hist[-1] if nll_hist else float("nan"))
        rho_last_local.append(rho_hist[-1] if rho_hist else float("nan"))
        cycle_times_local.append(time.perf_counter() - t_cycle_start)

        if show_progress:
            avg_step = sum(cycle_times_local) / len(cycle_times_local)
            iterator.set_postfix(rmse=f"{rmse_a:.4f}", rho=f"{rho_last_local[-1]:.3g}", step=f"{avg_step:.2f}s")

    result = {
        "obs_mode": obs_mode,
        "config": cfg_run,
        "n_requested": n_cycles,
        "n_completed": len(rmse_local),
        "rmse_truth_vs_analysis": np.asarray(rmse_local, dtype=np.float64),
        "rmse_analysis_obs": np.asarray(rmse_obs_local, dtype=np.float64),
        "rmse_analysis_unobs": np.asarray(rmse_unobs_local, dtype=np.float64),
        "spread_analysis": np.asarray(spread_local, dtype=np.float64),
        "obs_rmse": np.asarray(obs_rmse_local, dtype=np.float64),
        "neg_log_likelihood_last": np.asarray(nll_last_local, dtype=np.float64),
        "rho_last": np.asarray(rho_last_local, dtype=np.float64),
        "obs_pct_hist": np.asarray(obs_pct_local, dtype=np.float64),
        "cycle_times": np.asarray(cycle_times_local, dtype=np.float64),
        "final_rmse": float(rmse_local[-1]) if rmse_local else float("nan"),
        "mean_rmse": float(np.mean(rmse_local)) if rmse_local else float("nan"),
        "final_spread": float(spread_local[-1]) if spread_local else float("nan"),
        "mean_spread": float(np.mean(spread_local)) if spread_local else float("nan"),
    }
    if keep_history:
        result["hist_analysis"] = hist_local
        result["obs_mask_hist"] = obs_mask_local
        result["obs_flat_idx_fixed"] = obs_flat_idx_fixed_local.detach().cpu()
    return result


def run_parameter_sweep(obs_mode, parameter_sets, n_cycles, selection_metric="final_rmse"):
    rows = []
    results = []
    for run_idx, overrides in enumerate(parameter_sets, start=1):
        result = run_da_configuration(
            obs_mode,
            overrides=overrides,
            keep_history=False,
            show_progress=False,
            n_cycles=n_cycles,
        )
        results.append(result)
        row = {
            "run": run_idx,
            **overrides,
            "completed": result["n_completed"],
            "final_RMSE": result["final_rmse"],
            "mean_RMSE": result["mean_rmse"],
            "final_spread": result["final_spread"],
            "mean_spread": result["mean_spread"],
        }
        rows.append(row)
        print(
            f"run={run_idx} completed={result['n_completed']}/{n_cycles} "
            f"final_RMSE={result['final_rmse']:.6f} mean_RMSE={result['mean_rmse']:.6f} "
            f"override={overrides}"
        )

    table = pd.DataFrame(rows)
    metric_col = "final_RMSE" if selection_metric == "final_rmse" else "mean_RMSE"
    eligible = [
        (i, r) for i, r in enumerate(results)
        if r["n_completed"] == n_cycles and np.isfinite(r[selection_metric])
    ]
    if not eligible:
        raise RuntimeError("No sweep case completed all requested DA cycles.")

    best_i, best_result = min(
        eligible,
        key=lambda item: (
            item[1][selection_metric],
            item[1]["mean_rmse"],
        ),
    )
    table["completed_all"] = table["completed"] == n_cycles
    table = table.sort_values(
        ["completed_all", metric_col, "mean_RMSE"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return table, results, best_i, best_result


In [ ]:
# ----------------------------------------------------------------------
# Sweep setup: applies to any one of the four observation modes.
# ----------------------------------------------------------------------
SWEEP_CONFIG = OBS_MODE
SWEEP_N_DA_CYCLES = min(50, N_DA_CYCLES)  # set to N_DA_CYCLES for full-length tuning
SWEEP_SELECTION_METRIC = "final_rmse"      # or "mean_rmse"

# Keep the default grid compact. Expand the J_F factors/radius after the basic
# guidance scales are in a reasonable range.
SWEEP_PARAMETER_GRID = {
    "guidance_scale_lin": [4.0e-05, 5.0e-05],
    "guidance_scale_nonlin": [1.1e-04, 1.3e-04, 1.5e-04],
    "jf_scale_lin": [1.0],
    "jf_scale_nonlin": [1.0],
    "jf_local_radius": [1],
    "jf_weight_bandwidth_factor": [1.0],
    "residual_soften_alpha": [1.0],
    "latent_inflation": [1.0],
    "posterior_gamma": [1.0],
}
SWEEP_PARAMETER_SETS = make_parameter_grid(SWEEP_PARAMETER_GRID)

# Optional manual choice after inspecting the table.
# Example:
# BEST_SWEEP_OVERRIDE = {
#     "guidance_scale_lin": 5.0e-05,
#     "guidance_scale_nonlin": 1.3e-04,
#     "jf_scale_lin": 0.5,
#     "jf_scale_nonlin": 1.0,
#     "jf_local_radius": 1,
#     "jf_weight_bandwidth_factor": 1.0,
#     "residual_soften_alpha": 1.0,
#     "latent_inflation": 1.0,
#     "posterior_gamma": 1.0,
# }
BEST_SWEEP_OVERRIDE = None


In [ ]:
sweep_table, sweep_results, best_sweep_index, best_sweep_result = run_parameter_sweep(
    SWEEP_CONFIG,
    SWEEP_PARAMETER_SETS,
    n_cycles=SWEEP_N_DA_CYCLES,
    selection_metric=SWEEP_SELECTION_METRIC,
)
display(sweep_table)

best_override_auto = {
    key: best_sweep_result["config"][key]
    for key in (
        "guidance_scale_lin",
        "guidance_scale_nonlin",
        "jf_scale_lin",
        "jf_scale_nonlin",
        "jf_local_radius",
        "jf_weight_bandwidth_factor",
        "residual_soften_alpha",
        "latent_inflation",
        "posterior_gamma",
    )
}
print("Automatic best override:", best_override_auto)


In [ ]:
selected_override = best_override_auto if BEST_SWEEP_OVERRIDE is None else dict(BEST_SWEEP_OVERRIDE)

selected_result = run_da_configuration(
    SWEEP_CONFIG,
    overrides=selected_override,
    keep_history=True,
    show_progress=True,
    n_cycles=N_DA_CYCLES,
    desc=f"Selected best [{SWEEP_CONFIG}]",
)

print(
    f"Selected full run: completed={selected_result['n_completed']}/{N_DA_CYCLES}, "
    f"final_RMSE={selected_result['final_rmse']:.6f}, "
    f"mean_RMSE={selected_result['mean_rmse']:.6f}"
)


In [ ]:
# Unpack the selected full run so the original analysis/plot/save cells still work.
OBS_MODE = selected_result["obs_mode"]
active_cfg = selected_result["config"]

GUIDANCE_SCALE_LIN = float(active_cfg["guidance_scale_lin"])
GUIDANCE_SCALE_NONLIN = float(active_cfg["guidance_scale_nonlin"])
JF_SCALE_LIN = float(active_cfg["jf_scale_lin"])
JF_SCALE_NONLIN = float(active_cfg["jf_scale_nonlin"])
JF_LOCAL_RADIUS = int(active_cfg["jf_local_radius"])
JF_WEIGHT_BANDWIDTH_FACTOR = float(active_cfg["jf_weight_bandwidth_factor"])
JF_RIDGE_REL = float(active_cfg["jf_ridge_rel"])
JF_RIDGE_ABS = float(active_cfg["jf_ridge_abs"])
EDM_SIGMA_DATA_FALLBACK = float(active_cfg["edm_sigma_data_fallback"])
RESIDUAL_SOFTEN_ALPHA = float(active_cfg["residual_soften_alpha"])
latent_inflation = float(active_cfg["latent_inflation"])
posterior_gamma = float(active_cfg["posterior_gamma"])

obs_percent_fix = float(active_cfg.get("obs_percent_fix", 0.0))
obs_min = float(active_cfg.get("obs_min", np.nan))
obs_max = float(active_cfg.get("obs_max", np.nan))
K_NONOVERLAP = int(active_cfg.get("k_nonoverlap", OBS_MODE_CONFIGS["kstep_nonoverlap"]["k_nonoverlap"]))
N_BLOCKS = int(active_cfg.get("n_blocks", OBS_MODE_CONFIGS["block"]["n_blocks"]))
BLOCK_SHAPE = tuple(active_cfg.get("block_shape", OBS_MODE_CONFIGS["block"]["block_shape"]))
DIAGONAL_PERIODIC = bool(active_cfg.get("diagonal_periodic", OBS_MODE_CONFIGS["diagonal"]["diagonal_periodic"]))

hist_analysis = selected_result["hist_analysis"]
rmse_truth_vs_analysis = selected_result["rmse_truth_vs_analysis"].tolist()
rmse_analysis_obs = selected_result["rmse_analysis_obs"].tolist()
rmse_analysis_unobs = selected_result["rmse_analysis_unobs"].tolist()
spread_analysis = selected_result["spread_analysis"].tolist()
obs_rmse = selected_result["obs_rmse"].tolist()
neg_log_likelihood_last = selected_result["neg_log_likelihood_last"].tolist()
residual_rho_last = selected_result["rho_last"].tolist()
cycle_times = selected_result["cycle_times"].tolist()
obs_pct_hist = selected_result["obs_pct_hist"].tolist()
obs_mask_hist = selected_result["obs_mask_hist"]

n_da_cycles_completed = selected_result["n_completed"]

OBS_HAS_UNOBS = bool(np.isfinite(np.asarray(rmse_analysis_unobs, dtype=float)).any())


In [ ]:
print(
    f"Completed {n_da_cycles_completed}/{N_DA_CYCLES} DA cycles. "
    f"Last RMSE={rmse_truth_vs_analysis[-1] if rmse_truth_vs_analysis else float('nan'):.4f}; "
    f"mean rho(last sampler step)={np.nanmean(residual_rho_last) if residual_rho_last else float('nan'):.4g}"
)
if cycle_times:
    print(f"Avg time per cycle: {np.mean(cycle_times):.2f}s")


In [ ]:
# --- Result containers (normalized ensemble; physical conversion below) ---
path_analysis_ens = torch.stack([torch.stack(h, dim=0) for h in hist_analysis], dim=0)
path_analysis = path_analysis_ens.mean(dim=0)


In [ ]:
path_analysis_ens_phys = denormalize_tensor(path_analysis_ens, norm_stats)
path_analysis_np = path_analysis_ens_phys.detach().cpu().numpy()

rmse_analysis_np = np.asarray(rmse_truth_vs_analysis, dtype=np.float32)
rmse_analysis_obs_np = np.asarray(rmse_analysis_obs, dtype=np.float32)
rmse_analysis_unobs_np = np.asarray(rmse_analysis_unobs, dtype=np.float32)
spread_analysis_np = np.asarray(spread_analysis, dtype=np.float32)
rho_last_np = np.asarray(residual_rho_last, dtype=np.float32)
obs_rmse_np = np.asarray(obs_rmse, dtype=np.float32)
nll_last_np = np.asarray(neg_log_likelihood_last, dtype=np.float32)
cycle_times_np = np.asarray(cycle_times, dtype=np.float32)

obs_mask_hist_tensor = torch.stack(obs_mask_hist, dim=0)
obs_pct_hist_tensor = torch.tensor(obs_pct_hist, dtype=torch.float32)
obs_mask_hist_np = obs_mask_hist_tensor.numpy().astype("uint8")
obs_pct_hist_np = obs_pct_hist_tensor.numpy()


In [ ]:
# Selected-run outputs are unpacked above.


In [ ]:
# Physical ensemble path and NumPy metric arrays are prepared above.


In [ ]:
# Sweep table remains available as `sweep_table`.


In [ ]:
# Observation-mask and coverage histories are prepared above.


In [ ]:
# `rho_last_np` stores the mean relaxed residual norm at the final sampler step of each DA cycle.


In [ ]:
# Relaxed residual normalization replaces the previous diffusion-dependent observation denominator.


In [ ]:
def fmt_param(x):
    return f"{x:.7g}".replace(".", "p")

In [ ]:
def fmt_sci(x):
    mant, exp = f"{x:.2e}".split("e")
    mant = mant.rstrip("0").rstrip(".")
    exp = exp.replace("-", "_").replace("+", "")
    return mant.replace(".", "p") + "e" + exp

In [ ]:
obs_sigma_str = fmt_param(obs_sigma)
scale_lin_str = fmt_sci(GUIDANCE_SCALE_LIN)
scale_nonlin_str = fmt_sci(GUIDANCE_SCALE_NONLIN)
jf_lin_str = fmt_param(JF_SCALE_LIN)
jf_nonlin_str = fmt_param(JF_SCALE_NONLIN)
jf_bw_str = fmt_param(JF_WEIGHT_BANDWIDTH_FACTOR)
soft_alpha_str = fmt_param(RESIDUAL_SOFTEN_ALPHA)
obs_mode_str = str(OBS_MODE).replace("_", "")
alp_str = fmt_param(OBS_ALPHA)
lat_infl_str = fmt_param(latent_inflation)
post_infl_str = fmt_param(posterior_gamma)

case_name = (
    f"SQG_EnSF_MixedObs_SoftResNorm_LocalWeightedJF"
    f"_obsmode{obs_mode_str}"
    f"_Ncycles{N_DA_CYCLES}"
    f"_NE{NE}"
    f"_obssigma{obs_sigma_str}"
    f"_alpha{alp_str}"
    f"_softA{soft_alpha_str}"
    f"_scalelin{scale_lin_str}"
    f"_scalenonlin{scale_nonlin_str}"
    f"_jfLin{jf_lin_str}"
    f"_jfNonlin{jf_nonlin_str}"
    f"_jfR{JF_LOCAL_RADIUS}"
    f"_jfBW{jf_bw_str}"
)

if OBS_MODE in ("random_percent", "kstep_nonoverlap"):
    case_name += (
        f"_obsfix{fmt_param(obs_percent_fix)}"
        f"_obsrandmin{fmt_param(obs_min)}"
        f"_obsrandmax{fmt_param(obs_max)}"
    )
if OBS_MODE == "kstep_nonoverlap":
    case_name += f"_Knonoverlap{K_NONOVERLAP}"
elif OBS_MODE == "block":
    case_name += f"_nblocks{N_BLOCKS}_block{BLOCK_SHAPE[0]}x{BLOCK_SHAPE[1]}"
elif OBS_MODE == "diagonal":
    case_name += (
        f"_ndiag{active_cfg['n_diagonals_min']}to{active_cfg['n_diagonals_max']}"
        f"_periodic{int(DIAGONAL_PERIODIC)}"
    )


In [ ]:
filename_save = f"{case_name}_analysis.mat"

In [ ]:
save_folder = "../../output_DA/Figures/SQG_DA_1h/MixedObs"

In [ ]:
from scipy.io import savemat

In [ ]:
os.makedirs(save_folder, exist_ok=True)

In [ ]:
savemat(
    f"{save_folder}/{filename_save}",
    {
        "path_analysis": path_analysis_np,
        "rmse_analysis": rmse_analysis_np,
        "rmse_analysis_obs": rmse_analysis_obs_np,
        "rmse_analysis_unobs": rmse_analysis_unobs_np,
        "spread_analysis": spread_analysis_np,
        "rho_last": rho_last_np,
        "obs_rmse_all": obs_rmse_np,
        "neg_log_likelihood_last": nll_last_np,
        "cycle_times": cycle_times_np,
        "obs_mask_hist": obs_mask_hist_np,
        "obs_pct_hist": obs_pct_hist_np,
        "guidance_scale_lin": GUIDANCE_SCALE_LIN,
        "guidance_scale_nonlin": GUIDANCE_SCALE_NONLIN,
        "jf_scale_lin": JF_SCALE_LIN,
        "jf_scale_nonlin": JF_SCALE_NONLIN,
        "jf_local_radius": JF_LOCAL_RADIUS,
        "jf_weight_bandwidth_factor": JF_WEIGHT_BANDWIDTH_FACTOR,
        "jf_ridge_rel": JF_RIDGE_REL,
        "jf_ridge_abs": JF_RIDGE_ABS,
        "edm_sigma_data_fallback": EDM_SIGMA_DATA_FALLBACK,
        "residual_soften_alpha": RESIDUAL_SOFTEN_ALPHA,
        "latent_inflation": latent_inflation,
        "posterior_gamma": posterior_gamma,
    },
)


In [ ]:
print(net.sigma_max)

## Phase 5: Metrics vs assimilation cycle

In [ ]:
print(len(rmse_truth_vs_noda))

In [ ]:
n_done = len(rmse_truth_vs_analysis)
cycles = np.arange(n_done)
if n_done == 0:
    print("No completed DA cycles to plot.")
else:
    fig, axs = plt.subplots(4, 1, figsize=(9, 10), sharex=True)

    axs[0].plot(
        cycles, rmse_truth_vs_analysis, "o-", color="C0", lw=2, ms=5, alpha=0.95,
        label="analysis (+ obs)",
    )

    # if RUN_EXTRA_COMPARISONS:
    #     axs[0].plot(
    #         cycles, rmse_truth_vs_noda, "^-.", color="C2", lw=2, ms=5, alpha=0.95,
    #         label="no-DA (pure EDM)",
    #     )
    axs[0].set_ylabel("RMSE vs truth")
    axs[0].set_title("Ensemble-mean RMSE by case")
    axs[0].legend(fontsize=8, loc="best")
    axs[0].grid(True, alpha=0.35)

    axs[1].plot(
        cycles, rmse_analysis_obs, "o-", color="C3", lw=2.2, ms=4, alpha=0.95,
        label="analysis — observed sites",
    )

    if OBS_HAS_UNOBS:
        axs[1].plot(
            cycles, rmse_truth_vs_analysis, color="C0", lw=1.5, alpha=0.3,
            label="analysis (full grid)",
        )

        axs[1].plot(
            cycles, rmse_analysis_unobs, "s-", color="C4", lw=2.2, ms=4, alpha=0.95,
            label="analysis — unobserved sites",
        )
        axs[1].set_title("Analysis RMSE: observed vs unobserved mask")
    else:
        axs[1].set_title("Analysis RMSE (full observation)")
    axs[1].set_ylabel("RMSE vs truth")
    axs[1].legend(fontsize=8, loc="best")
    axs[1].grid(True, alpha=0.35)

    axs[2].plot(cycles, spread_analysis, "o-", color="C0", lw=2, ms=4, alpha=0.9, label="analysis spread")
    
    axs[2].set_ylabel("ens. spread")
    axs[2].legend(fontsize=8)
    axs[2].grid(True, alpha=0.35)

    axs[3].plot(cycles, obs_rmse[:n_done], "o-", color="0.45", lw=2, ms=4, alpha=0.9, label="obs RMSE")
    axs[3].set_ylabel("obs RMSE")
    axs[3].set_xlabel("assimilation cycle (after burn-in)")
    axs[3].legend(fontsize=8)
    axs[3].grid(True, alpha=0.35)

    extra = " + no-DA" if RUN_EXTRA_COMPARISONS else ""
    plt.suptitle(f"DA ({obs_kind}, alpha={OBS_ALPHA}), NE={NE}: {n_done} cycles{extra}")
    plt.tight_layout()
    plt.show()

## Animate: truth vs ensemble member(s)

One row per member in `ANIM_ENS_MEMBERS`. Columns: truth | optional no-DA / bgrd | analysis.



In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Animation settings ---
ANIM_CHANNEL = 0
ANIM_ENS_MEMBERS = [0,1] 
ANIM_MAX_FRAMES = 80

t0 = path_burn.shape[0] - 1
truth_seg = path_truth_full[t0:, ANIM_CHANNEL]
T_seg = min(
    truth_seg.shape[0],
    path_analysis_ens.shape[1] - t0,
    # path_bgrd_ens.shape[1] - t0,
    ANIM_MAX_FRAMES,
)
if RUN_EXTRA_COMPARISONS and path_noda_ens is not None:
    T_seg = min(T_seg, path_noda_ens.shape[1] - t0)

case_labels = ["truth"]
case_getters = [lambda _k, ts: truth_seg[:T_seg]]
# if RUN_EXTRA_COMPARISONS and path_noda_ens is not None:
#     case_labels.append("no-DA")
#     case_getters.append(
#         lambda k, ts: ens_channel_phys(path_noda_ens, k, ts, ANIM_CHANNEL, norm_stats)
#     )
# case_labels.extend(["bgrd", "analysis"])
case_labels.extend(["analysis"])
case_getters.extend(
    [
        # lambda k, ts: ens_channel_phys(path_bgrd_ens, k, ts, ANIM_CHANNEL, norm_stats),
        lambda k, ts: ens_channel_phys(path_analysis_ens, k, ts, ANIM_CHANNEL, norm_stats),
    ]
)
n_cols = len(case_labels)
n_rows = len(ANIM_ENS_MEMBERS)
ts = slice(t0, t0 + T_seg)

stacked = []
for getter in case_getters:
    stacked.append(getter(ANIM_ENS_MEMBERS[0], ts)[:T_seg])
stacked = torch.stack(stacked, dim=0)
vmin, vmax = float(stacked.min()), float(stacked.max())

fig, axs = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.0 * n_rows), squeeze=False)
fig.subplots_adjust(left=0.02, right=0.98, top=0.88, bottom=0.02, hspace=0.15, wspace=0.06)
ims = []
for r, k in enumerate(ANIM_ENS_MEMBERS):
    row_data = [getter(k, ts)[:T_seg] for getter in case_getters]
    for c in range(n_cols):
        ax = axs[r, c]
        im = ax.imshow(
            row_data[c][0].numpy(), cmap="RdBu_r", vmin=vmin, vmax=vmax, animated=True
        )
        ims.append((row_data[c], im))
        title = case_labels[c] if n_rows == 1 else f"{case_labels[c]} ens {k}"
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
title = fig.suptitle("", fontsize=11)


def update(frame_k):
    for row_data, im in ims:
        im.set_array(row_data[frame_k].numpy())
    title.set_text(f"post-burn-in step {frame_k}  |  ch={ANIM_CHANNEL}")
    return [im for _, im in ims] + [title]


anim = FuncAnimation(fig, update, frames=T_seg, interval=200, blit=False, repeat=True)
plt.close(fig)
HTML(anim.to_jshtml())



### Pointwise error animation

Same layout as the static error plot, animated over post-burn-in time steps.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Error animation settings ---

ERR_ANIM_CHANNEL = 0
ERR_ANIM_MEMBER = 0
ERR_ANIM_MAX_FRAMES = 80

t0 = path_burn.shape[0] - 1
T_err = min(
    path_truth_full.shape[0] - t0,
    path_analysis_ens.shape[1] - t0,
    # path_bgrd_ens.shape[1] - t0,
    ERR_ANIM_MAX_FRAMES,
)
# if RUN_EXTRA_COMPARISONS and path_noda_ens is not None:
#     T_err = min(T_err, path_noda_ens.shape[1] - t0)

truth_err_seg = path_truth_full[t0 : t0 + T_err, ERR_ANIM_CHANNEL]
ts_err = slice(t0, t0 + T_err)
k = ERR_ANIM_MEMBER

err_cases = [("analysis", path_analysis_ens)]
# if RUN_EXTRA_COMPARISONS and path_noda_ens is not None:
#     err_cases.insert(0, ("no-DA", path_noda_ens))
# err_cases.insert(-1, ("bgrd", path_bgrd_ens))

err_seg = []
err_vmax_anim = 0.0
for label, path_ens in err_cases:
    pred = ens_channel_phys(path_ens, k, ts_err, ERR_ANIM_CHANNEL, norm_stats)
    err = (pred - truth_err_seg).abs()
    err_seg.append((label, err))
    err_vmax_anim = max(err_vmax_anim, float(err.max()))
err_vmax_anim = max(err_vmax_anim, 1e-12)

n_err_cols = len(err_seg)
fig, axs = plt.subplots(1, n_err_cols, figsize=(3.4 * n_err_cols, 3.2), squeeze=False)
fig.subplots_adjust(left=0.04, right=0.90, top=0.82, bottom=0.06, wspace=0.12)

err_ims = []
for c, (label, err) in enumerate(err_seg):
    ax = axs[0, c]
    im = ax.imshow(err[0].numpy(), origin="lower", cmap="hot", vmin=0, vmax=err_vmax_anim, animated=True)
    ax.set_title(f"{label} ens {k}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    err_ims.append((err, im))

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.68])
fig.colorbar(im, cax=cbar_ax, label="|PV error|")
err_title = fig.suptitle("", fontsize=11)


def update_err(frame_k):
    for err, im in err_ims:
        im.set_array(err[frame_k].numpy())
    err_title.set_text(f"post-burn-in step {frame_k}  |  ch={ERR_ANIM_CHANNEL}  |  ens {k}")
    return [im for _, im in err_ims] + [err_title]


err_anim = FuncAnimation(fig, update_err, frames=T_err, interval=200, blit=False, repeat=True)
plt.close(fig)
HTML(err_anim.to_jshtml())

